# HAMBEE · EVA4
Pulsa **▶** en la única celda y, si aparece el selector, carga el ZIP que acompaña al notebook. El modelo ONNX de la época 37 procesa automáticamente las 7.140 imágenes y genera los gráficos en CPU.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from zipfile import ZipFile
from IPython.display import FileLink, Markdown, display
import matplotlib.pyplot as plt

bundle = Path("EVA4_RHAGUER_YMALDONADO.zip").resolve()
if not bundle.is_file():
    from google.colab import files
    files.upload()
if not bundle.is_file():
    raise FileNotFoundError(f"Carga {bundle.name} desde la carpeta que acompaña al notebook.")
work = Path("hambee-eva4-colab").resolve()
work.mkdir(exist_ok=True)
with ZipFile(bundle) as archive:
    for item in archive.infolist():
        if not (work / item.filename).resolve().is_relative_to(work):
            raise ValueError("El ZIP contiene una ruta inválida.")
        archive.extract(item, work)
project = work / "EVA4"
runtime = work / ".venv"
python = runtime / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
if not python.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(runtime)], check=True)
subprocess.run([str(python), "-m", "pip", "install", "-q", "-r",
                str(project / "requirements-test.txt")], check=True)

def run(script, *args):
    result = subprocess.run([str(python), script, *map(str, args)], cwd=project,
                            capture_output=True, text=True, encoding="utf-8")
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout

model = json.loads((project / "models/metadata.json").read_text(encoding="utf-8"))
assert model["model_version"] == "1.1.0" and model["best_epoch"] == 37
print("Verificando el contrato de FastAPI…")
contract = json.loads(run("smoke.py", "validacion/test/P/P10109-127r.jpg"))
out = work / "resultados"
out.mkdir(exist_ok=True)
print("Procesando las 7.140 imágenes…")
full = json.loads(run("procesar_dataset.py", "data/Dataset.zip", "--output", out))
assert full["processed"] == 7140 and full["test"]["correct"] == 70
print("Midiendo la latencia y los tamaños de lote…")
run("benchmark.py", "validacion", "--output", out/"benchmark.json")
run("benchmark_lotes.py", "validacion", "--output", out/"lotes.json")
metrics = json.loads((out/"benchmark.json").read_text())
batches = json.loads((out/"lotes.json").read_text())
assert metrics["identical_predictions"] and batches["identical_predictions"]
opt, base = metrics["optimizado"], metrics["sin_optimizacion"]
assert opt["confusion_np_p"] == [[34, 0], [1, 36]]
display(Markdown(f'''### Resultados de esta sesión
- **Modelo 1.1.0 · época 37 de 40**
- **Prueba:** {full["test"]["correct"]} de 71 ({opt["accuracy"]:.2%}) · F1 macro {opt["macro_f1"]:.4f}
- **Dataset completo:** {full["processed"]} imágenes en {full["processing_seconds"]:.2f} s · {full["processing_images_per_second"]:.1f} imágenes/s
- **FastAPI:** respuestas {", ".join(map(str, contract["validated_status_codes"]))}
'''))
fig, grid = plt.subplots(2, 2, figsize=(12, 7.4), layout="constrained")
latency, matrix, rate, history = grid.ravel()
for offset, info, color, label in [(-.17, base, "#80928b", "Grafo base"), (.17, opt, "#266d53", "Optimizado")]:
    bars = latency.bar([offset, 1+offset], [info["p50_ms"],info["p95_ms"]], .30, color=color, label=label)
    latency.bar_label(bars, fmt="%.2f", padding=3)
latency.set_xticks([0,1],["p50","p95"])
latency.set(title="Latencia del modelo",ylabel="Milisegundos",ylim=(0,max(base["p95_ms"],opt["p95_ms"])*1.3))
latency.legend(frameon=False)
values = opt["confusion_np_p"]
matrix.imshow(values,cmap="Greens",vmin=0,vmax=40)
matrix.set_xticks([0,1],["NP","P"]); matrix.set_yticks([0,1],["NP","P"])
matrix.set(title="Matriz de confusión",xlabel="Predicha",ylabel="Real")
for i, row in enumerate(values):
    for j, value in enumerate(row):
        matrix.text(j,i,str(value),ha="center",va="center",color="white" if value>20 else "black")
sizes = [x["batch_size"] for x in batches["batch_metrics"]]
rate.plot(sizes,[x["images_per_second"] for x in batches["batch_metrics"]],marker="o",color="#266d53")
rate.set(title="Capacidad por lote",xlabel="Imágenes por lote",ylabel="Imágenes/s",xticks=sizes)
training = model["history"]
history.semilogy([r["epoch"] for r in training],[r["val_loss"] for r in training],color="#266d53")
history.scatter(37,training[36]["val_loss"],color="#b37c17",label="Época 37")
history.set(title="Pérdida de validación",xlabel="Época",ylabel="Entropía cruzada")
history.legend(frameon=False)
plt.show()
csv_path = out / "predicciones-dataset.csv"
try:
    from google.colab import files
    import ipywidgets as widgets
    download = widgets.Button(description="Descargar CSV")
    download.on_click(lambda _: files.download(str(csv_path)))
    display(download)
except ImportError:
    display(FileLink(str(csv_path.relative_to(Path.cwd()))))
